# Setup and GPU placement

## Prerequisites

Participants should know basic C/C++, CUDA kernels, and MPI point-to-point calls. They need Gadi access and membership of project `vp91`.

## Why combine MPI and CUDA?

CUDA accelerates computation within one node and MPI connects processes across distributed-memory nodes. Combining them lets an application solve problems too large for one GPU, use several GPUs in parallel, or extend an MPI application beyond one node. Each MPI process has a private address space and is identified by a rank. Processes communicate with point-to-point operations such as `MPI_Send` and `MPI_Recv`; collective operations such as `MPI_Reduce` combine values from several ranks.

The basic execution pattern is:

```cpp
MPI_Init(&argc, &argv);
MPI_Comm_rank(MPI_COMM_WORLD, &rank);
MPI_Comm_size(MPI_COMM_WORLD, &size);
/* application work and MPI communication */
MPI_Finalize();
```

The MPI compiler wrapper supplies the required headers and libraries, while `mpirun` or the PBS-launched equivalent starts one process for each rank. This workshop uses MPI for distributed coordination and CUDA kernels for local grid computation.

Start in a clone on a filesystem visible to compute nodes, then submit:

```console
qsub jobs-scripts/01-rank-device.pbs
qstat -swx <job-id>
less cuda-mpi-map.o<job-id>
```

The script requests two GPUs and two MPI ranks. `01-rank-device.cu` forms a shared-memory communicator with `MPI_Comm_split_type`. Its node-local rank selects a CUDA device. Expected output maps local ranks 0 and 1 to different GPUs on the same host. This follows the one-MPI-process-per-device pattern while using a node-local rank rather than world rank so that the mapping remains valid across nodes.

## How each MPI rank gets a different GPU

The example code is `src/01-rank-device.cu`, and the matching launcher is `jobs-scripts/01-rank-device.pbs`. A rank should not blindly use its global MPI rank as a GPU number. Global rank 2 on a second node is not the same as GPU 2 on the first node.

![MPI world ranks are split into node-local communicators, then each local rank selects a GPU](../docs/source/images/01-select-device-local-communicator.png)

The key call is:

```cpp
// Each MPI rank must pick a different CUDA device on the same node.
// The shared-memory communicator groups ranks that are on the same host,
// so their local rank values are 0, 1, 2, ... in order. The mapping is:
//   device = local_rank % device_count
// With a 2-rank, 2-GPU job, rank 0 gets local rank 0 and rank 1 gets local
// rank 1, so the first process uses GPU 0 and the second uses GPU 1.
int device = select_device(MPI_COMM_WORLD, &local_rank);
```

The helper in `src/00-common.h` does the actual mapping:

```cpp
MPI_Comm local;
MPI_CHECK(MPI_Comm_split_type(world, MPI_COMM_TYPE_SHARED, 0, MPI_INFO_NULL,
                              &local));
MPI_Comm_rank(local, &local_rank);
cudaGetDeviceCount(&devices);
int device = local_rank % devices;
cudaSetDevice(device);
```

`MPI_COMM_TYPE_SHARED` creates a communicator containing only ranks on the same node. Those ranks are numbered from 0 upward as local ranks. The actual GPU choice is:

```cpp
device = local_rank % device_count
```

With two GPUs and two MPI ranks, the mapping is `0 -> GPU 0` and `1 -> GPU 1`. Across multiple nodes, world ranks remain unique, but the GPU number should be chosen from the node-local rank. This keeps one MPI process on each GPU without conflicts.

## Why local rank matters

The mapping between MPI ranks and node-local GPU devices depends on how `mpirun` is configured and how the batch system places processes. A world rank of 2 does not necessarily mean GPU 2; it may be the first rank on a second node, or it may be placed with other ranks on the same host. The code must therefore be robust to different MPI launch configurations rather than assuming rank number and device number are the same.

The scheduler and `mpirun` may distribute global ranks differently across hosts, but each host still numbers ranks in its shared-memory communicator from zero:

![Two valid MPI host mappings show that global ranks can be distributed differently while local ranks restart at zero on each host](../docs/source/images/02-why-local-rank-required.png)

The helper uses a node-local rank obtained from `MPI_COMM_TYPE_SHARED`. This makes device selection independent of global rank ordering and ensures each process picks a GPU consistent with its placement. The modulo fallback keeps the code valid when ranks and visible GPUs are not exactly matched, but oversubscribing multiple ranks onto one GPU is normally undesirable. Prefer one rank per requested GPU when the job layout allows it.

## Exercise (10 minutes)

Change the script to request one GPU and run one rank. Predict and verify the local rank and device. Then explain why merely setting `CUDA_VISIBLE_DEVICES` globally is insufficient for several ranks.

## PBS anatomy

The job script describes the resources reserved for the run and the MPI command. Important directives are:

- `-P vp91`: project.
- `-q gpuvolta`: GPU queue.
- `-l ncpus=24`: 24 CPU cores.
- `-l ngpus=2`: two GPUs, matching one process per GPU.
- `-l mem=8gb`: memory limit.
- `-l jobfs=1GB`: local temporary disk space.
- `-l walltime=00:10:00`: maximum runtime.
- `-l storage=scratch/jxj900+gdata/vp91`: project storage.
- `-l wd`: start in the submission directory.
- `-N cuda-mpi-map`: readable scheduler name.

The final command uses Open MPI rank placement:

```console
mpirun -np 2 --map-by ppr:2:node build/bin/01-rank-device
```

`ppr` means processes per resource. `ppr:2:node` places two MPI ranks on each node. Here both ranks are placed on one node and can select different local GPUs. `ppr:1:node` would place each rank on a separate node, which is a different distribution for multi-node jobs. Explicit mapping is useful because CUDA-aware MPI examples assume rank placement matches the GPU layout.

In [ ]:
%%bash
set -e
cd "$(git rev-parse --show-toplevel)"
cmake -S . -B build -G Ninja -DCMAKE_CUDA_ARCHITECTURES=70
cmake --build build --target 01-rank-device

In [ ]:
%%bash
set -e
cd "$(git rev-parse --show-toplevel)"
qsub jobs-scripts/01-rank-device.pbs